In [1]:
import logging
import sys
import os

logging.basicConfig(stream=sys.stdout, level=logging.INFO)
logging.getLogger().addHandler(logging.StreamHandler(stream=sys.stdout))

In [2]:
import os
os.environ['NUMEXPR_MAX_THREADS'] = '4'
os.environ['NUMEXPR_NUM_THREADS'] = '2'
import numexpr as ne

In [9]:
!pip install -r requirements.txt

In [32]:
import tiktoken
from llama_index.core.settings import Settings
from llama_index.core.callbacks import CallbackManager, TokenCountingHandler

token_counter = TokenCountingHandler(
    tokenizer=tiktoken.encoding_for_model("text-embedding-ada-002").encode,
    verbose=True
)
callback_manager = CallbackManager([token_counter])
Settings.callback_manager = callback_manager


In [18]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, StorageContext, Settings, load_index_from_storage
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

# Set the embedding model once, globally
Settings.embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-small-en-v1.5"
)

PERSIST_DIR = './storage/cache/andrew/sleep'
try:
  storage_context = StorageContext.from_defaults(persist_dir=PERSIST_DIR)
  index = load_index_from_storage(storage_context)
  print('Loaded index from cache')
except():
  documents = SimpleDirectoryReader('assets/AndrewHuberman/sleep').load_data()
  index = VectorStoreIndex.from_documents(documents)
  index.storage_context.persist(persist_dir=PERSIST_DIR)

print(token_counter.total_embedding_token_count)


# try:
#     storage_context = StorageContext.from_defaults(persist_dir=PERSIST_DIR)
#     index = load_index_from_storage(storage_context)
#     print('Loaded index from cache')
# except (FileNotFoundError, ValueError):
#     print('No cache found, building index...')
#     documents = SimpleDirectoryReader('assets/AndrewHuberman/sleep').load_data()
#     index = VectorStoreIndex.from_documents(documents, show_progress=True)
#     index.storage_context.persist(persist_dir=PERSIST_DIR)
#     print('Index persisted to disk')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded index from cache
0


In [40]:
from google.colab import userdata
from llama_index.llms.huggingface_api import HuggingFaceInferenceAPI

import tiktoken
from llama_index.core.settings import Settings
from llama_index.core.callbacks import CallbackManager, TokenCountingHandler

from transformers import AutoTokenizer

llama_tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B-Instruct",
                                                 token=userdata.get('HF_TOKEN'))

token_counter = TokenCountingHandler(
    tokenizer=llama_tokenizer.encode,
    verbose=True
)

# token_counter = TokenCountingHandler(
#     tokenizer=tiktoken.encoding_for_model("text-embedding-ada-002").encode,
#     verbose=True
# )
callback_manager = CallbackManager([token_counter])
Settings.callback_manager = callback_manager

model = "meta-llama/Llama-3.1-8B-Instruct"
Settings.llm = HuggingFaceInferenceAPI(
    model_name= model,
    token=userdata.get('HF_TOKEN'),
    callback_manager=callback_manager
)

qe = index.as_query_engine(response_mode='compact')
response = qe.query("How does sleep enhance learning memory?")
print(f'Embedding tokens: {token_counter.total_embedding_token_count}\n \
       LLM Prompts: {token_counter.prompt_llm_token_count}\n \
       LLM completitions: {token_counter.completion_llm_token_count}\n \
       Total LLM token count: {token_counter.total_llm_token_count}')
print(response)

print(qe.llm)
print(Settings.llm.callback_manager)
print(Settings.llm.callback_manager.handlers)

OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct.
403 Client Error. (Request ID: Root=1-69b02d0a-5efedad6085c879874844aaf;25037c8d-5441-43ed-82fa-5f282aa0c7e6)

Cannot access gated repo for url https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct/resolve/main/config.json.
Access to model meta-llama/Llama-3.1-8B-Instruct is restricted and you are not in the authorized list. Visit https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct to ask for access.

In [6]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, StorageContext, load_index_from_storage

documents = SimpleDirectoryReader('assets/AndrewHuberman/sleep').load_data()
index = VectorStoreIndex.from_documents(documents)

KeyboardInterrupt: 

In [ ]:
query_engine = index.as_query_engine()


In [13]:
from llama_index.core.retrievers  import VectorIndexRetriever
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.indices.postprocessor import SimilarityPostprocessor, KeywordNodePostprocessor
from llama_index.core import get_response_synthesizer

retriever = VectorIndexRetriever(
    index=index,
    similarity_top_k=4
)

s_processor = SimilarityPostprocessor(
    similarity_cutoff=0.83
)

k_processor = KeywordNodePostprocessor(
    required_keywords=['supplement']
)

response_synthesizer = get_response_synthesizer(
    response_mode='compact',
)

query_engine = RetrieverQueryEngine(
    retriever=retriever,
    node_postprocessors=[k_processor, s_processor],
    response_synthesizer=response_synthesizer
)
    # vector_store_query_mode='default'

ValueError: 
******
Could not load OpenAI model. If you intended to use OpenAI, please check your OPENAI_API_KEY.
Original error:
No API key found for OpenAI.
Please set either the OPENAI_API_KEY environment variable or openai.api_key prior to initialization.
API keys can be found or created at https://platform.openai.com/account/api-keys

******

In [ ]:
from llama_index.core.prompts import PromptTemplate

text_qa_template_str = (
    "Context information is below.\n"
    "---------------------\n"
    "{context_str}\n"
    "---------------------\n"
    "Using both the context information and also using your own knowledge, "
    "answer the question: {query_str}\n"
    "If the context isn't helpful, you can also answer the question on your own.\n"
)

text_qa_template = PromptTemplate(text_qa_template_str)

In [ ]:
from llama_index.llms.huggingface_api import HuggingFaceInferenceAPI
from google.colab import userdata


Settings.llm = HuggingFaceInferenceAPI(
    model_name="HuggingFaceH4/zephyr-7b-beta",
    token=userdata.get('HF_TOKEN')
)

response = index.as_query_engine(
    text_qa_template=text_qa_template
).query("How does sleep enhance learning memory?")

print(response)

During sleep, specifically during the phases of slow wave sleep and REM sleep, certain patterns of brain activity facilitate the learning and encoding of procedural information and emotional memories, as well as the release of growth hormone promotes physical and mental health. Dr. Gina Poe, a sleep researcher at UCLA, explains that the timing and duration of sleep play crucial roles in both aspects. Getting enough sleep, including a consistent sleep schedule ensures adequate growth hormone release, which is essential for overall health and longevity. Consumption of electrolytes such as magnesium and potassium, crucial for neuronal function, and sleep before or after exercise can enhance learning and memory consolidation. Element LMNT offers a free sample pack at lmnt.com.


In [ ]:
text_qa_template_str = (
    "You are an Andrew huberman assistant that can read Andrew Huberman podcast notes.\n"
    "Always answer the query only using the provided context information, "
    "and not prior knowledge.\n"
    "Some rules to follow:\n"
    "1. Never directly reference the given context in your answer.\n"
    "2. Avoid statements like 'Based on the context, ...' or "
    "'The context information ...' or anything along "
    "those lines."
    "Context information is below.\n"
    "---------------------\n"
    "{context_str}\n"
    "---------------------\n"
    "Answer the question: {query_str}\n"
)

text_qa_template = PromptTemplate(text_qa_template_str)

In [ ]:
print(response)

During sleep, specifically during the phases of slow wave sleep and REM sleep, certain patterns of brain activity facilitate the learning and encoding of procedural information and emotional memories, as well as the release of growth hormone promotes physical and mental health. Dr. Gina Poe, a sleep researcher at UCLA, explains that the timing and duration of sleep play crucial roles in both aspects. Getting enough sleep, including a consistent sleep schedule ensures adequate growth hormone release, which is essential for overall health and longevity. Consumption of electrolytes such as magnesium and potassium, crucial for neuronal function, and sleep before or after exercise can enhance learning and memory consolidation. Element LMNT offers a free sample pack at lmnt.com.


In [ ]:
print(response)

During sleep, specifically during the phases of slow wave sleep and REM sleep, certain patterns of brain activity facilitate the learning and encoding of procedural information and emotional memories, as well as the release of growth hormone promotes physical and mental health. Dr. Gina Poe, a sleep researcher at UCLA, explains that the timing and duration of sleep play crucial roles in both aspects. Getting enough sleep, including a consistent sleep schedule ensures adequate growth hormone release, which is essential for overall health and longevity. Consumption of electrolytes such as magnesium and potassium, crucial for neuronal function, and sleep before or after exercise can enhance learning and memory consolidation. Element LMNT offers a free sample pack at lmnt.com.


In [ ]:
print(response)

During sleep, specifically during the phases of slow wave sleep and REM sleep, certain patterns of brain activity facilitate the learning and encoding of procedural information and emotional memories, as well as the release of growth hormone promotes physical and mental health. Dr. Gina Poe, a sleep researcher at UCLA, explains that the timing and duration of sleep play crucial roles in both aspects. Getting enough sleep, including a consistent sleep schedule ensures adequate growth hormone release, which is essential for overall health and longevity. Consumption of electrolytes such as magnesium and potassium, crucial for neuronal function, and sleep before or after exercise can enhance learning and memory consolidation. Element LMNT offers a free sample pack at lmnt.com.


In [ ]:
print(response.source_nodes)

[NodeWithScore(node=TextNode(id_='6baf522d-846a-4ced-a6d1-f33f6d743e04', embedding=None, metadata={'file_path': '/content/assets/AndrewHuberman/sleep/05_Understanding_and_Using_Dreams_to_Learn_and_to_Forget_Huberman_Lab_Podcast_5.txt', 'file_name': '05_Understanding_and_Using_Dreams_to_Learn_and_to_Forget_Huberman_Lab_Podcast_5.txt', 'file_type': 'text/plain', 'file_size': 75844, 'creation_date': '2026-03-10', 'last_modified_date': '2026-03-10'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='220e680b-0bee-467b-815f-34bf70be469f', node_type='4', metadata={'file_path': '/content/assets/AndrewHuberman/sleep/05_Understanding_and_Using_Dreams_to_Learn_and_to_Forget_Huberman_Lab_Podcast_5.txt', 'file_name': '05

In [ ]:
from llama_index.core.response.pprint_utils import pprint_response
pprint_response(response, show_source=True)

Final Response: During sleep, specifically during the phases of slow
wave sleep and REM sleep, certain patterns of brain activity
facilitate the learning and encoding of procedural information and
emotional memories, as well as the release of growth hormone promotes
physical and mental health. Dr. Gina Poe, a sleep researcher at UCLA,
explains that the timing and duration of sleep play crucial roles in
both aspects. Getting enough sleep, including a consistent sleep
schedule ensures adequate growth hormone release, which is essential
for overall health and longevity. Consumption of electrolytes such as
magnesium and potassium, crucial for neuronal function, and sleep
before or after exercise can enhance learning and memory
consolidation. Element LMNT offers a free sample pack at lmnt.com.
______________________________________________________________________
Source Node 1/2
Node ID: 6baf522d-846a-4ced-a6d1-f33f6d743e04
Similarity: 0.7953482640413583
Text: Now this isn't always cognitiv